### Setup Environment

In [1]:
!git clone https://github.com/NhuGiap04/Fk-Diffusion-Steering.git

Cloning into 'Fk-Diffusion-Steering'...
remote: Enumerating objects: 357, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 357 (delta 97), reused 87 (delta 58), pack-reused 206 (from 3)
Receiving objects: 100% (357/357), 211.02 MiB | 39.06 MiB/s, done.
Resolving deltas: 100% (183/183), done.


In [1]:
%cd Fk-Diffusion-Steering
!git pull

/content/Fk-Diffusion-Steering
Already up to date.


In [2]:
%pip install -r requirements.txt -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Building editable for diffusers (pyproject.toml) ... done


### Evolve Steering

In [2]:
%cd text_to_image

import os
# os.environ["CUDA_VISIBLE_DEVICES"] = ''
# os.environ['HF_HOME'] = ''

import json
import argparse
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from PIL import Image

import torch
from diffusers import DDIMScheduler

/content/Fk-Diffusion-Steering/text_to_image


In [16]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from diffusers import DDIMScheduler
from evolve_diffusers import BaseSDXL

device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = BaseSDXL.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from evolve_diffusers.steer_pipeline import iterative_sample_with_stein


def iterative_stein_loop(
    model,
    prompt,
    *,
    num_loops=4,
    num_particles=4,
    steer_start_timestep=200,
    steer_end_timestep=20,
    base_threshold=0.0,
    stein_step_size=0.04,
    stein_bandwidth=None,
    stein_rejected_penalty=0.0,
    num_inference_steps=50,
    guidance_scale=5.0,
    guidance_reward_fn="ImageReward",
    max_images_to_show=4,
):
    """Run iterative Stein steering and display loop-level diagnostics + final images."""
    out = iterative_sample_with_stein(
        model=model,
        prompt=prompt,
        num_loops=num_loops,
        num_particles=num_particles,
        steer_start_timestep=steer_start_timestep,
        steer_end_timestep=steer_end_timestep,
        base_threshold=base_threshold,
        stein_step_size=stein_step_size,
        stein_bandwidth=stein_bandwidth,
        stein_rejected_penalty=stein_rejected_penalty,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        guidance_reward_fn=guidance_reward_fn,
    )

    mean_rewards = [float(r.mean().item()) for r in out["rewards"]]
    accepted_counts = [int(a.shape[0]) for a in out["accepted"]]
    thresholds = out["thresholds"]
    loop_idx = np.arange(1, len(mean_rewards) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(loop_idx, mean_rewards, marker="o", label="mean reward")
    axes[0].plot(loop_idx, thresholds, marker="s", label="threshold")
    axes[0].set_xlabel("Loop")
    axes[0].set_ylabel("Score")
    axes[0].set_title("Reward and threshold per loop")
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    axes[1].bar(loop_idx, accepted_counts)
    axes[1].set_xlabel("Loop")
    axes[1].set_ylabel("# accepted particles")
    axes[1].set_title("Accepted particles per loop")
    axes[1].grid(axis="y", alpha=0.3)
    fig.tight_layout()
    plt.show()

    final_images = out["results"][-1].images
    n = min(max_images_to_show, len(final_images))
    fig, ax = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        ax = [ax]

    for i in range(n):
        ax[i].imshow(final_images[i])
        ax[i].axis("off")
        ax[i].set_title(f"Final loop sample {i + 1}")

    fig.suptitle("Iterative Stein loop: final generated samples", fontsize=13)
    fig.tight_layout()
    plt.show()

    return out


if "pipe" not in globals():
    raise RuntimeError("Run the steer_sample setup cell first so `pipe` is defined.")

loop_out = iterative_stein_loop(
    model=pipe,
    prompt="a photo of a brown knife and a blue donut",
    num_loops=4,
    num_particles=4,
    steer_start_timestep=400,
    steer_end_timestep=100,
    stein_step_size=0.01,
    stein_rejected_penalty=0.0,
    guidance_scale=5.0,
)

print("Best mean reward:", loop_out["best_mean_reward"])
print("Best reward:", loop_out["best_reward"])
print("Std reward:", loop_out["std_reward"])
print("Num loops returned:", len(loop_out["results"]))